In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from blox2.utils import make_scaled_trajectory

plt.rcParams.update({
    "font.size": 16,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16,
})

all_path = "../data/zinc_dft/properties.csv"
# all_path = "../data/material/properties.csv"
n_obs_0 = 10

dir_names = [
    "dft_multi/01-29_180034_lgb/seed_init_0",
]

legends = None
legends = [
    "LGB",
]

trajectories = []
for i, dn in enumerate(dir_names):
    init_path = f"results/{dn}/initial_observed_properties.csv"
    obs_path = f"results/{dn}/observation_history.csv"
    t = make_scaled_trajectory(init_path, obs_path, all_path)
    # t = make_scaled_trajectory(init_path, obs_path, None) # No scaling (Don't use this for scores)
    trajectories.append(t)
    if legends:
        plt.scatter(t[:, 0], t[:, 1], label=legends[i], alpha=0.5, s=1)
    else:
        plt.scatter(t[:, 0], t[:, 1], label=dn, alpha=0.5, s=1)

plt.legend(markerscale=5)

In [ ]:
# occupancy
from blox2 import make_scaler, occupancy_trajectory

bins = 50

all_props = pd.read_csv(all_path).to_numpy()
scaler = make_scaler(all_props, d=2)
X_all = scaler.transform(all_props)

for i, t in enumerate(trajectories):
    ch_trj = occupancy_trajectory(t, X_all=X_all, bins=bins)
    if legends:
        plt.plot(ch_trj[n_obs_0:], label=legends[i])
    else:
        plt.plot(ch_trj[n_obs_0:], label=dir_names[i])
        
plt.xlabel("Number of samplings")
plt.ylabel(f"Occupancy (bins: {bins})")
plt.rcParams.update({"legend.fontsize": 16})
plt.legend()

In [ ]:
# convex hull area
from blox2 import convex_hull_area_trajectory

for i, t in enumerate(trajectories):
    ch_trj = convex_hull_area_trajectory(t)
    if legends:
        plt.plot(ch_trj[n_obs_0:], label=legends[i])
    else:
        plt.plot(ch_trj[n_obs_0:], label=dir_names[i])

# plt.xscale("log", base=10)
# plt.yscale("log", base=10)
plt.rcParams.update({"legend.fontsize": 10})
plt.xlabel("Number of samplings")
plt.ylabel("Convex hull area")
plt.legend()

In [ ]:
# Alpha concave hull area (takes some time)
from blox2 import alpha_concave_hull_area_trajectory

for i, t in enumerate(trajectories):
    ch_trj = alpha_concave_hull_area_trajectory(t, alpha=1)
    if legends:
        plt.plot(ch_trj[n_obs_0:], label=legends[i])
    else:
        plt.plot(ch_trj[n_obs_0:], label=dir_names[i])

# plt.xscale("log", base=10)
# plt.yscale("log", base=10)
plt.rcParams.update({"legend.fontsize": 10})
plt.xlabel("Number of samplings")
plt.ylabel("Alpha convex hull area")
plt.legend()

In [ ]:
# Stein discrepancy
from blox2 import stein_discrepancy_trajectory

sigma = 1

# fig = plt.gcf()
# fig.set_size_inches(12, 9)

for i, t in enumerate(trajectories):
    ch_trj = stein_discrepancy_trajectory(t, sigma=sigma)
    if legends:
        plt.plot(ch_trj[n_obs_0:], label=legends[i])
    else:
        plt.plot(ch_trj[n_obs_0:], label=dir_names[i])

# plt.xscale("log", base=10)
plt.yscale("log", base=10)
plt.rcParams.update({"legend.fontsize": 10})
plt.xlabel("Number of samplings")
plt.ylabel(f"Stein discrepancy (σ={sigma})")
plt.legend()

In [ ]:
# convex hull perimeter
from blox2 import convex_hull_perimeter_trajectory

for i, t in enumerate(trajectories):
    ch_trj = convex_hull_perimeter_trajectory(t)
    if legends:
        plt.plot(ch_trj[n_obs_0:], label=legends[i])
    else:
        plt.plot(ch_trj[n_obs_0:], label=dir_names[i])

# plt.xscale("log", base=10)
# plt.yscale("log", base=10)
plt.rcParams.update({"legend.fontsize": 16})
plt.xlabel("Number of samplings")
plt.ylabel("Convex hull perimeter")
plt.legend()

In [ ]:
from blox2 import plot_buffer_union

plot_buffer_union(trajectories[0], radius=0.5, resolution=4)

In [ ]:
import alphashape
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon

alpha = 0.0075
alpha_shape = alphashape.alphashape(trajectories[0], alpha=alpha)
fig, ax = plt.subplots()
ax.scatter(trajectories[0][:,0], trajectories[0][:,1], s=1)
geom = alpha_shape

if geom.geom_type == "Polygon":
    polys = [geom]
elif geom.geom_type == "MultiPolygon":
    polys = list(geom.geoms)
else:
    raise ValueError(f"alpha_shape is {geom.geom_type}, not Polygon. Try a smaller alpha.")

for poly in polys:
    x, y = poly.exterior.xy
    ax.fill(x, y, alpha=0.2)

plt.title(f"α concave (α={alpha})")
plt.show()